## 0. Importações

Parte inicial, configuração de ambiente, importações e ajustes gerais. Principais ferramentas utilizdas:
- Python: Linguagem utilizada
- Pandas: Biblioteca de análise de dados
- Numpy: Dependência para a biblioteca pandas e utilitários para processo
- Json: Biblioteca para lindar com campos em formatação json

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import re
import gdown
import shutil
import os

from google.colab import drive
from pathlib import Path

In [ ]:
pip install pyarrow fastparquet

In [ ]:
drive.mount('/content/drive')
GOOGLE_DRIVE_PATH = '/content/drive/MyDrive/Plenário Virtual'

RAW_PATH = Path('data/raw/ArquivosConcatenados.csv')
PROCESSED_PATH = Path('data/processed')

print(f"Raw data path: {RAW_PATH}")
print(f"Processed data path: {PROCESSED_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw data path: data/raw/ArquivosConcatenados.csv
Processed data path: data/processed


## 1. Visualização inicial

Objetivos:
- Identificar natureza dos dados
- Estabelecer uma descrição precisa dos dados
- Identificar inconsistências (se for o caso)
- Realizar limpeza (se for o caso)
- Preparar dados para etapadas posteriores

### 1.1 Importação do dataset e visualização inicial

In [ ]:
from src.cleaning import load_raw

# Importação do dataset
df = load_raw(Path(GOOGLE_DRIVE_PATH) / 'data/raw/ArquivosConcatenados.csv')

# Visualização básica
df

In [ ]:
# Colunas do dataset
df.columns.unique()

Index(['incidente', 'classe', 'nome_processo', 'classe_extenso',
       'tipo_processo', 'liminar', 'origem', 'relator', 'autor1',
       'len(partes_total)', 'partes_total', 'data_protocolo', 'origem_orgao',
       'lista_assuntos', 'resumo', 'len(andamentos_lista)', 'andamentos_lista',
       'len(decisões)', 'decisões', 'len(deslocamentos)',
       'deslocamentos_lista', 'status_processo'],
      dtype='object')

#### Descrição das variáveis do conjunto de dados

A tabela abaixo descreve cada uma das 21 colunas presentes no dataset, que contém informações processuais de ações judiciais (principalmente ADI e ADO) do Supremo Tribunal Federal (STF).

| Variável | Descrição |
|----------|-----------|
| `incidente` | Número de identificação do processo no sistema (aparentemente um código numérico único). |
| `classe` | Sigla da classe processual (ex.: ADI, ADO). |
| `nome_processo` | Nome resumido do processo, geralmente composto pela classe seguida de um número sequencial. |
| `classe_extenso` | Nome completo da classe processual (ex.: "AÇÃO DIRETA DE INCONSTITUCIONALIDADE"). |
| `tipo_processo` | Indica se o processo tramita em meio físico ou eletrônico. |
| `liminar` | Lista com os tipos de tutela de urgência requeridos (ex.: "MEDIDA LIMINAR", "TUTELA PROVISÓRIA"). Quando vazio (`[]`), indica ausência de pedido liminar. |
| `origem` | Sigla da unidade da federação de origem do processo (ex.: RO, DF, SP, MG, PA). |
| `relator` | Nome do ministro relator do processo no STF. |
| `autor1` | Nome da primeira parte autora (pessoa física, jurídica, órgão público ou entidade). |
| `len(partes_total)` | Número total de partes envolvidas no processo (autores, réus, interessados, etc.). |
| `data_protocolo` | Data de protocolo (entrada) do processo no tribunal, no formato `dd/mm/aaaa`. |
| `origem_orgao` | Órgão ou unidade judiciária de onde se originou o processo (ex.: "FÓRUM DA COMARCA DE RANCHARIA", "SUPREMO TRIBUNAL FEDERAL"). |
| `lista_assuntos` | Lista de assuntos/cadastros temáticos atribuídos ao processo (ex.: "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO"). |
| `len(andamentos_lista)` | Número de eventos (andamentos) registrados na movimentação do processo. |
| `andamentos_lista` | Lista de dicionários contendo detalhes dos andamentos, como índice, data, nome do ato, descrição, etc. |
| `len(decisões)` | Quantidade de decisões judiciais proferidas no processo. |
| `decisões` | Lista de dicionários com informações das decisões, incluindo índice, data, nome da decisão, tipo e conteúdo. |
| `len(deslocamentos)` | Número de deslocamentos (movimentações físicas ou eletrônicas) do processo entre unidades ou órgãos. |
| `deslocamentos_lista` | Lista de dicionários com dados dos deslocamentos, como data de recebimento, data de envio, origem, destino, etc. |
| `status_processo` | Situação atual do processo: "Finalizado" ou "Em andamento". |

> **Observação:** As variáveis com prefixo `len()` representam a contagem de elementos das respectivas listas (andamentos, decisões, deslocamentos). As listas estão estruturadas no formato JSON, conforme exemplos na tabela HTML.

### 1.2 Explorando os valores únicos das variáveis

#### Incidentes

In [ ]:
quantidade_valores_unicos_incidente = df['incidente'].unique()
quantidade_valores_unicos_incidente = len(quantidade_valores_unicos_incidente)
print(f"A quantidade de valores únicos para incidentes é de: {quantidade_valores_unicos_incidente}")

A quantidade de valores únicos para incidentes é de: 9358


Podemos observar que a quantidade de valores únicos do incidente é igual a quantidade de linhas do dataset (9358) que são a quantidade de processos. Portanto, um incidente' referência diretamente um processo.

#### Classe

In [ ]:
quantidade_valores_unicos_classe = df['classe'].unique()
print(f"A quantiadde de valores únicos para classe é de: {quantidade_valores_unicos_classe}")

A quantiadde de valores únicos para classe é de: ['ADC' 'ADI' 'ADO' 'ADPF']


O dataset apresenta especificamente as seguintes classes:
- ADC: Ação Declaratória de Constitucionalidade
- ADI: Ação Direta de Inconstitucionalidade
- ADO: Ação Direta de Inconstitucionalidade por Omissão
- ADPF: Arguição de Descumprimento de Preceito Fundamental

#### Nome processo

In [ ]:
quantidade_valores_unicos_nome_processo = df['nome_processo'].unique()
quantidade_valores_unicos_nome_processo = len(quantidade_valores_unicos_nome_processo)
print(f"A quantidade de valores únicos para nome_processo é de: {quantidade_valores_unicos_nome_processo}")

# Descobrindo o range de cada processo
df[['classe', 'numero']] = df['nome_processo'].str.split(' ', n=1, expand=True)
df['numero'] = pd.to_numeric(df['numero'])
ranges = df.groupby('classe')['numero'].agg(['min', 'max']).reset_index()
print(ranges)


A quantidade de valores únicos para nome_processo é de: 9358
  classe  min   max
0    ADC    1   100
1    ADI    1  7941
2    ADO    1    94
3   ADPF    1  1311


Podemos identificar os seguintes ranges para cada processo:
- ADC: 1 - 100
- ADI: 1 - 7941
- ADO: 1 - 94
- ADPF: 1 - 1311

#### Origem

In [ ]:
# Analisando a distribuição por Unidade da Federação
contagem_origem = df['origem'].value_counts(dropna=False)
display(contagem_origem)

print(f"\nQuantidade de UFs/Origens distintas: {len(df['origem'].unique())}")

,count
origem,
DF,3759
RJ,536
SP,487
SC,349
RS,311
PR,302
MG,275
ES,261
GO,230



Quantidade de UFs/Origens distintas: 30


A análise da coluna `origem` permite identificar de onde surgem as principais controvérsias constitucionais. O DF costuma liderar por ser a sede de muitos órgãos federais, mas é possível observar a representatividade de estados como SP, RJ e MG.

também possível observar a ausência de valores.

#### Relator

In [ ]:
print(df['relator'].unique())

print(f"\nTotal de relatores distintos registrados: {len(df['relator'].dropna().unique())}")

['AYRES BRITTO' 'GILMAR MENDES' 'JOAQUIM BARBOSA' 'ROSA WEBER'
 'CÁRMEN LÚCIA' 'CEZAR PELUSO' 'EDSON FACHIN' 'CELSO DE MELLO'
 'MARCO AURÉLIO' 'MOREIRA ALVES' 'DIAS TOFFOLI' 'RICARDO LEWANDOWSKI'
 'EROS GRAU' 'LUIZ FUX' 'CARLOS VELLOSO' 'ALEXANDRE DE MORAES'
 'NELSON JOBIM' 'LUÍS ROBERTO BARROSO' 'SYDNEY SANCHES' 'CRISTIANO ZANIN'
 nan 'MAURÍCIO CORRÊA' 'NUNES MARQUES' 'NÉRI DA SILVEIRA'
 'SEPÚLVEDA PERTENCE' 'ILMAR GALVÃO' 'ELLEN GRACIE' 'FRANCISCO REZEK'
 'OCTAVIO GALLOTTI' 'PAULO BROSSARD' 'MENEZES DIREITO'
 'MINISTRO PRESIDENTE' 'TEORI ZAVASCKI' 'ALDIR PASSARINHO' 'CÉLIO BORJA'
 'ANDRÉ MENDONÇA' 'FLÁVIO DINO' 'CARLOS MADEIRA']

Total de relatores distintos registrados: 37


A lista de relatores inclui tanto ministros atuais quanto aposentados, o que é esperado dado que o dataset abrange processos desde a redemocratização (1988). Isso ajuda a entender o volume de acervo histórico por gabinete.

#### Autor Principal (autor1)

In [ ]:
autor = df['autor1'].unique()
print(autor)

print(f"\nQuantidade de autores distintos (autor1): {len(df['autor1'].unique())}")

# Verificando os maiores proponentes de ações
principais_autores = df['autor1'].value_counts().head(10)
display(principais_autores)

print(f"\nQuantidade total de autores distintos (autor1): {len(df['autor1'].unique())}")

['DINETE LESSA' 'GOVERNADOR DO DISTRITO FEDERAL'
 'ASSOCIAÇÃO DOS MAGISTRADOS BRASILEIROS - AMB' ...
 'SINDICATO NACIONAL DAS EMPRESAS DE ENCOMENDAS EXPRESSAS'
 'ASSOCIACAO BRASILEIRA DE PROTEINA ANIMAL'
 'ARTICULAÇÃO DOS POVOS E ORGANIZAÇÕES INDÍGENAS DO BRASIL - APIB']

Quantidade de autores distintos (autor1): 1498


,count
autor1,
PROCURADOR-GERAL DA REPÚBLICA,1784
PARTIDO DOS TRABALHADORES - PT,183
CONSELHO FEDERAL DA ORDEM DOS ADVOGADOS DO BRASIL,168
PARTIDO DEMOCRÁTICO TRABALHISTA - PDT,164
GOVERNADOR DO ESTADO DE SÃO PAULO,144
ASSOCIAÇÃO DOS MAGISTRADOS BRASILEIROS - AMB,142
GOVERNADOR DO ESTADO DE SANTA CATARINA,139
GOVERNADOR DO ESTADO DO RIO GRANDE DO SUL,130
PARTIDO SOCIALISTA BRASILEIRO - PSB,130



Quantidade total de autores distintos (autor1): 1498


O campo `autor1` revela os principais 'players' do controle concentrado de constitucionalidade, como o Procurador-Geral da República, Governadores e partidos políticos. A grande variedade de autores únicos reflete a ampla legitimidade ativa prevista na Constituição de 1988.

#### Tipo de Processo

In [ ]:
quantidade_valores_unicos_tipo_processo = df['tipo_processo'].unique()
print(f"A quantidade de valores únicos para tipo_processo é de: {quantidade_valores_unicos_tipo_processo}")

# Quantiade entre físico e eletrónico
tipos_proc = df['tipo_processo'].value_counts()
display(tipos_proc)

A quantidade de valores únicos para tipo_processo é de: ['Físico' 'Eletrônico']


,count
tipo_processo,
Eletrônico,5618
Físico,3740


Temos a presensa de dois tipos de processos: físico e eletrônico. Aparentemte a quantiade dp eletrônico tem predominância sobre a quantidade do físico

#### Data de Protocolo

In [ ]:
# Analisando o range temporal e datas com mais protocolos
protocolos = pd.to_datetime(df['data_protocolo'], dayfirst=True, errors='coerce')
print(f"Data mais antiga: {protocolos.min()}")
print(f"Data mais recente: {protocolos.max()}")

print("\nTop 5 datas com maior volume de protocolos:")
display(df['data_protocolo'].value_counts().head(5))

Data mais antiga: 1988-10-06 00:00:00
Data mais recente: 2026-03-06 00:00:00

Top 5 datas com maior volume de protocolos:


,count
data_protocolo,
23/11/2022,42
02/04/1990,29
24/11/2023,29
17/06/2013,28
03/05/2021,28


A variável de data permite entender o fluxo histórico de entrada de ações no tribunal. O range cobre desde a promulgação da Constituição de 1988 até os dias atuais, com picos que podem estar correlacionados a eventos políticos ou mudanças legislativas.

#### Status do Processo

In [ ]:
# Verificando a situação atual dos processos no dataset
status_unicos = df['status_processo'].value_counts()
display(status_unicos)

,count
status_processo,
Finalizado,8335
Em andamento,1023


Esta variável é binária e fundamental para separar o acervo histórico (Finalizado) do acervo vivo (Em andamento) do STF, permitindo focar análises de produtividade ou tempo de tramitação.

#### Variáveis de Contagem (len)

In [ ]:
# Explorando a complexidade através das contagens
contagens = ['len(partes_total)', 'len(andamentos_lista)', 'len(decisões)']
display(df[contagens].describe())

,len(partes_total),len(andamentos_lista),len(decisões)
count,9358.000000,9358.000000,9358.000000
mean,7.585381,53.980445,3.220987
std,10.852028,73.367250,4.444990
min,0.000000,0.000000,0.000000
25%,3.000000,27.000000,1.000000
50%,5.000000,43.000000,2.000000
75%,8.000000,63.000000,4.000000
max,472.000000,3247.000000,137.000000


#### Órgão de Origem (origem_orgao)

In [ ]:
# Todos os orgãos de origem envolvidos
orgaos_envolvidos = df['origem_orgao'].unique()
for i in orgaos_envolvidos:
  print(f"{i}")

print(f"\nQuantidade de órgãos de origem distintos: {len(df['origem_orgao'].unique())}")

SUPREMO TRIBUNAL FEDERAL
PROCURADORIA-GERAL DE JUSTICA DO ESTADO DO PI
nan
FÓRUM DA COMARCA DE RANCHARIA
TRIBUNAL DE JUSTIÇA ESTADUAL
TRIBUNAL REGIONAL FEDERAL
TRIBUNAL REGIONAL ELEITORAL
JUIZ ELEITORAL
NÃO IDENTIFICADO
JUIZ DE DIREITO
SUPERIOR TRIBUNAL DE JUSTIÇA
JUIZ DO TRABALHO
CONSELHO NACIONAL DE JUSTIÇA
TRIBUNAL DE CONTAS DA UNIÃO
MINISTÉRIO PÚBLICO FEDERAL
3ª VARA DA JUSTIÇA FEDERAL DO DISTRITO FEDERAL
TRIBUNAL DE JUSTIÇA DO ESTADO DE ALAGOAS
CONSELHO NACIONAL DO MINISTÉRIO PÚBLICO
TRIBUNAL REGIONAL FEDERAL DA 1ª REGIÃO
TRIBUNAL SUPERIOR ELEITORAL
TRIBUNAL REGIONAL DO TRABALHO DA 13ª REGIÃO
TRIBUNAL REGIONAL FEDERAL DA 4ª REGIÃO
TRIBUNAL DE JUSTIÇA DO ESTADO DE SÃO PAULO
TRIBUNAL DE JUSTIÇA DO ESTADO DA BAHIA
ADVOCACIA-GERAL DA UNIÃO
ÓRGÃO/ENTE DA ADMINISTRAÇÃO
TRIBUNAL REGIONAL DO TRABALHO
TJGO - 2ª TURMA RECURSAL DA 3ª REGIÃO - ANÁPOLIS
JUIZ FEDERAL

Quantidade de órgãos de origem distintos: 29


Temos 29 orgãos envolvidos na origem dos processos no total

#### Pedidos de Liminar

In [ ]:
liminares = df['liminar'].unique()
print(liminares)

TypeError: unhashable type: 'list'

#### Assuntos Cadastrados (lista_assuntos)

In [ ]:
print(df['lista_assuntos'].unique())

['["DIREITO CIVIL | Coisas | Enfiteuse"]'
 '["DIREITO PROCESSUAL CIVIL E DO TRABALHO | Liquidação / Cumprimento / Execução | Efeito Suspensivo / Impugnação / Embargos à Execução"]'
 '["DIREITO PROCESSUAL CIVIL E DO TRABALHO | Órgãos Judiciários e Auxiliares da Justiça", "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Agentes Políticos | Magistratura | Afastamento"]'
 ...
 '["QUESTÕES DE ALTA COMPLEXIDADE, GRANDE IMPACTO E REPERCUSSÃO | COVID-19", "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Domínio Público", "DIREITO CIVIL | Coisas | Posse"]'
 '["DIREITO PROCESSUAL CIVIL E DO TRABALHO | Liquidação / Cumprimento / Execução | Precatório | Sequestro de Verbas Públicas", "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Orçamento | Repasse de Verbas Públicas", "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Organização Político-administrativa / Administração Pública | Município"]'
 '["DIREITO DO TRABALHO | Direito Individual do T

In [ ]:
# Visualizando a diversidade de temas abordados
assuntos_unicos = df['lista_assuntos'].value_counts().head(10)
display(assuntos_unicos)

,count
lista_assuntos,
"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Controle de Constitucionalidade""]",721
"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Controle de Constitucionalidade | Processo Legislativo""]",416
[],413
"[""ASSUNTO PARA PROCESSO ANTIGO | | PROCESSO ANTIGO""]",199
"[""ASSUNTOS DIVERSOS""]",152
"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Controle de Constitucionalidade | Inconstitucionalidade Material""]",96
"[""DIREITO TRIBUTÁRIO | Impostos | ICMS/ Imposto sobre Circulação de Mercadorias""]",75
"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Organização Político-administrativa / Administração Pública""]",63
"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Controle de Constitucionalidade"", ""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO | Organização Político-administrativa / Administração Pública""]",57


### 1.3 Estrutura Interna: Andamentos, Decisões e Deslocamentos

#### Visualição detalhada de um processo complexo
Abaixo, apresento um exemplo do conteúdo de um único registro para cada uma das variáveis de lista para entendermos o esquema de dados (chaves e valores).

In [ ]:
# Encontrando um incidente complexo (com muitos andamentos e decisões)
# Vamos somar as colunas de 'len' para achar um caso extremo
df['complexidade_total'] = df['len(andamentos_lista)'] + df['len(decisões)'] + df['len(deslocamentos)']
incidente_complexo = df.sort_values('complexidade_total', ascending=False).iloc[0]

def exibir_json_completo(objeto_str, titulo):
    try:
        # Converte a string (ajustando aspas se necessário) para objeto Python
        # Usamos ast.literal_eval por segurança com as aspas simples do dataset original
        import ast
        dados = ast.literal_eval(objeto_str)
        print(f"=== {titulo} (Total de itens: {len(dados)}) ===")
        # Exibimos apenas os 3 primeiros para não travar o log, mas com a estrutura completa
        print(json.dumps(dados[:3], indent=4, ensure_ascii=False))
        print("\n... (lista truncada para visualização) ...\n")
    except Exception as e:
        print(f"Erro ao processar {titulo}: {e}")

print(f"EXEMPLO DE PROCESSO COMPLEXO: {incidente_complexo['nome_processo']} (Incidente: {incidente_complexo['incidente']})\n")

EXEMPLO DE PROCESSO COMPLEXO: ADPF 854 (Incidente: 6199750)



In [ ]:
exibir_json_completo(incidente_complexo['partes_total'], "PARTES")

=== PARTES (Total de itens: 31) ===
[
    {
        "_index": 1,
        "tipo": "REQTE.(S)",
        "nome": "PARTIDO SOCIALISMO E LIBERDADE - PSOL"
    },
    {
        "_index": 2,
        "tipo": "ADV.(A/S)",
        "nome": "RAPHAEL SODRE CITTADINO (5742-A/AP, 53229/DF, 435368/SP)"
    },
    {
        "_index": 3,
        "tipo": "ADV.(A/S)",
        "nome": "BRUNA DE FREITAS DO AMARAL (69296/DF)"
    }
]

... (lista truncada para visualização) ...



In [ ]:
exibir_json_completo(incidente_complexo['andamentos_lista'], "ANDAMENTOS")

=== ANDAMENTOS (Total de itens: 3247) ===
[
    {
        "index": 3247,
        "data": "06/03/2026",
        "nome": "Certidão",
        "complemento": "Certifico que, até o dia 27/02/2026, quanto ao item 11, I e II da decisão de 27/01/2026, não se manifestou o Município de Areia Branca - RN.",
        "julgador": "NA",
        "validade": "valid",
        "link": "https://portal.stf.jus.br/processos/downloadPeca.asp?id=15384762095&ext=.pdf",
        "link_tipo": "",
        "link_conteúdo": "Supremo Tribunal Federal\nCertidão\nCertifico que, até o dia 27/02/2026, quanto ao item 11, I e II da decisão de\n27/01/2026, não se manifestou o Município de Areia Branca - RN.\nBrasília, 6 de março de 2026.\nProcessos Originários Cíveis\nDocumento assinado digitalmente\nDocumento assinado digitalmente conforme MP n° 2.200-2/2001 de 24/08/2001. O documento pode ser acessado pelo endereço\nhttp://www.stf.jus.br/portal/autenticacao/autenticarDocumento.asp sob o código 02E0-4F83-0602-494C e senha 

In [ ]:
exibir_json_completo(incidente_complexo['decisões'], "DECISÕES")

=== DECISÕES (Total de itens: 137) ===
[
    {
        "index": 3160,
        "data": "03/03/2026",
        "nome": "Convertido em diligência",
        "complemento": "Sem Descrição",
        "julgador": "MIN. FLÁVIO DINO",
        "validade": "valid",
        "link": "https://portal.stf.jus.br/processos/downloadPeca.asp?id=15384622930&ext=.pdf",
        "link_tipo": "",
        "link_conteúdo": "ARGUIÇÃO DE DESCUMPRIMENTO DE PRECEITO FUNDAMENTAL 854\nDISTRITO FEDERAL\nRELATOR :MIN. FLÁVIO DINO\nREQTE.(S) :PARTIDO SOCIALISMO E LIBERDADE - PSOL\nADV.(A/S) :RAPHAEL SODRE CITTADINO\nADV.(A/S) :BRUNA DE FREITAS DO AMARAL\nADV.(A/S) :PRISCILLA SODRÉ PEREIRA\nINTDO.(A/S) :PRESIDENTE DA REPÚBLICA\nPROC.(A/S)(ES) :ADVOGADO-GERAL DA UNIÃO\nINTDO.(A/S) :CONGRESSO NACIONAL\nPROC.(A/S)(ES) :ADVOGADO-GERAL DA UNIÃO\nINTDO.(A/S) :SENADO FEDERAL\nPROC.(A/S)(ES) :ADVOGADO-GERAL DA UNIÃO\nADV.(A/S) :ADVOGADO DO SENADO FEDERAL\nINTDO.(A/S) :CÂMARA DOS DEPUTADOS\nPROC.(A/S)(ES) :ADVOGADO-GERAL DA UNIÃO\n

In [ ]:
exibir_json_completo(incidente_complexo['deslocamentos_lista'], "DESLOCAMENTOS")

=== DESLOCAMENTOS (Total de itens: 318) ===
[
    {
        "index": 318,
        "data_recebido": "Recebido em 04/03/2026",
        "enviado por": "GERÊNCIA DE PROCESSOS ORIGINÁRIOS CÍVEIS",
        "recebido por": "Enviado por GERÊNCIA DE COMUNICAÇÕES PROCESSUAIS em 04/03/2026",
        "guia": "Guia 3595/2026"
    },
    {
        "index": 317,
        "data_recebido": "Recebido em 03/03/2026",
        "enviado por": "GERÊNCIA DE COMUNICAÇÕES PROCESSUAIS",
        "recebido por": "Enviado por GERÊNCIA DE PROCESSOS ORIGINÁRIOS CÍVEIS em 03/03/2026",
        "guia": "Guia 5316/2026"
    },
    {
        "index": 316,
        "data_recebido": "Recebido em 09/02/2026",
        "enviado por": "GERÊNCIA DE PROCESSOS ORIGINÁRIOS CÍVEIS",
        "recebido por": "Enviado por GERÊNCIA DE COMUNICAÇÕES PROCESSUAIS em 09/02/2026",
        "guia": "Guia 1855/2026"
    }
]

... (lista truncada para visualização) ...



#### Descrição Técnica das Variáveis Internas (JSON)

Com base na extração do processo **ADPF 854**, aqui está o mapeamento do que cada campo dentro das listas representa:

#### 1. Partes (`partes_total`)
| Chave | Descrição |
| :--- | :--- |
| `_index` | Identificador sequencial da parte no processo. |
| `tipo` | Papel jurídico (ex: REQTE para Requerente, INTDO para Interessado, AM. CURIAE para Amicus Curiae). |
| `nome` | Nome da pessoa, órgão ou entidade, seguido muitas vezes pela OAB no caso de advogados. |

#### 2. Andamentos (`andamentos_lista`)
| Chave | Descrição |
| :--- | :--- |
| `index` | Número do andamento (ordem cronológica inversa). |
| `data` | Data em que o evento foi registrado. |
| `nome` | Título do ato processual (ex: Petição, Conclusos, Expedido). |
| `complemento` | Texto descritivo detalhando o ato (ex: número da petição ou destino de um ofício). |
| `julgador` | Identifica se houve um órgão julgador específico no ato (geralmente 'NA' em atos administrativos). |
| `link` | URL para o documento em PDF (quando disponível). |
| `link_conteúdo` | Texto extraído via OCR do documento vinculado ao andamento. |

#### 3. Decisões (`decisões`)
| Chave | Descrição |
| :--- | :--- |
| `index` | Vínculo com o índice da lista de andamentos. |
| `data` | Data da prolação ou publicação da decisão. |
| `nome` | Tipo de decisão (ex: Medida Cautelar, Decisão Monocrática, Julgamento Virtual). |
| `julgador` | Autor da decisão (Ministro Relator ou Colegiado/Plenário). |
| `link_conteúdo` | O teor completo da decisão transcrito (fundamental para análise de texto/NLP). |

#### 4. Deslocamentos (`deslocamentos_lista`)
| Chave | Descrição |
| :--- | :--- |
| `index` | Identificador da movimentação física/eletrônica. |
| `data_recebido` | Data em que o destino acusou o recebimento do processo. |
| `enviado por` | Unidade de origem da remessa. |
| `recebido por` | Unidade de destino que recebeu a carga. |
| `guia` | Número da Guia de Remessa para rastreio administrativo. |

Com esses exemplos, podemos observar que:
- **Andamentos:** Contêm o histórico completo de passos processuais.
- **Decisões:** Focam nos atos decisórios, muitas vezes incluindo links para o teor da decisão.
- **Deslocamentos:** Registram o trâmite físico ou eletrônico entre gabinetes e secretarias.

### 1.4 Análise descritiva do dataset

In [ ]:
# Obtendo uma análise descritiva do dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9358 entries, 0 to 9357
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   incidente              9358 non-null   int64 
 1   classe                 9358 non-null   object
 2   nome_processo          9358 non-null   object
 3   classe_extenso         9358 non-null   object
 4   tipo_processo          9358 non-null   object
 5   liminar                9358 non-null   object
 6   origem                 9335 non-null   object
 7   relator                9287 non-null   object
 8   autor1                 9351 non-null   object
 9   len(partes_total)      9358 non-null   int64 
 10  partes_total           9358 non-null   object
 11  data_protocolo         9358 non-null   object
 12  origem_orgao           9178 non-null   object
 13  lista_assuntos         9358 non-null   object
 14  resumo                 9358 non-null   object
 15  len(andamentos_lista)

#### Dimensões
- **Total de linhas:** 9.358
- **Total de colunas:** 24

#### Tipos de dados por coluna

| Tipo   | Quantidade | Colunas                                                                 |
|--------|------------|-------------------------------------------------------------------------|
| `int64`| 7          | `incidente`, `len(partes_total)`, `len(andamentos_lista)`, `len(decisões)`, `len(deslocamentos)`, `numero`, `complexidade_total` |
| `str/obj`| 17       | Todas as demais (incluindo campos que armazenam listas em formato JSON) |

#### Valores ausentes (non-null vs total)

| Coluna               | Não-nulos | Nulos | % Nulos | Observação                                     |
|----------------------|-----------|-------|---------|------------------------------------------------|
| `origem`             | 9.335     | 23    | 0,25%   | Poucos registros sem informação de UF          |
| `relator`            | 9.287     | 71    | 0,76%   | Ministros relatores não identificados          |
| `autor1`             | 9.351     | 7     | 0,07%   | Raríssimos casos sem autor principal           |
| `origem_orgao`       | 9.178     | 180   | 1,92%   | Maior lacuna; órgão de origem não informado    |
| **Demais colunas**   | 9.358     | 0     | 0%      | Nenhum valor faltante                          |

> A coluna `origem_orgao` continua sendo a que apresenta maior número de ausências (180 linhas), o que é comum em processos de controle concentrado que nascem no próprio STF ou de migrações de sistemas antigos.

#### Detalhamento de colunas com conteúdo semi-estruturado

O dataset utiliza strings que representam listas de objetos para armazenar a complexidade do processo:

- `partes_total`: Nomes e tipos de todos os envolvidos.
- `andamentos_lista`: Histórico completo de eventos processuais.
- `decisões`: Conteúdo e links das decisões proferidas.
- `deslocamentos_lista`: Trâmite entre departamentos do tribunal.

**Observação sobre Complexidade:** A nova variável `complexidade_total` foi criada somando as extensões das listas de andamentos, decisões e deslocamentos para identificar os processos com maior volume de dados, como a **ADPF 854**.

### 1.5 Verificação de nulos e duplicados

#### Nulos

In [ ]:
# Mostra a quantidade absoluta de nulos em cada coluna
print("Valores Nulos Absolutos:")
print(df.isnull().sum())

# BÔNUS: Mostra o percentual (%) de nulos em cada coluna (ótimo para o relatório)
print("\nPercentual de Nulos (%):")
print((df.isnull().sum() / len(df)) * 100)

Valores Nulos Absolutos:
incidente                  0
classe                     0
nome_processo              0
classe_extenso             0
tipo_processo              0
liminar                    0
origem                    23
relator                   71
autor1                     7
len(partes_total)          0
partes_total               0
data_protocolo             0
origem_orgao             180
lista_assuntos             0
resumo                     0
len(andamentos_lista)      0
andamentos_lista           0
len(decisões)              0
decisões                   0
len(deslocamentos)         0
deslocamentos_lista        0
status_processo            0
numero                     0
complexidade_total         0
dtype: int64

Percentual de Nulos (%):
incidente                0.000000
classe                   0.000000
nome_processo            0.000000
classe_extenso           0.000000
tipo_processo            0.000000
liminar                  0.000000
origem                   0.245779
re

#### Duplicados

In [ ]:
# Para verificar duplicatas em um DataFrame com listas, precisamos converter as colunas de listas para string
total_linhas_duplicadas = df.astype(str).duplicated().sum()
print(f"Total de linhas completamente duplicadas: {total_linhas_duplicadas}")

Total de linhas completamente duplicadas: 0


## 2. Limpeza

Inicialmente criando uma copia do dataset por precaução

In [ ]:
from src.cleaning import clean

proc = clean(df)

### 2.1 Padronização
Nesta seção, realizamos o tratamento de valores nulos, conversão de tipos

#### Padronização de nulos

#### Transformação do `nome_processo`


In [ ]:
proc['numero_processo'].value_counts

<bound method IndexOpsMixin.value_counts of 0        10
1        11
2        12
3        13
4        14
       ... 
9353    973
9354    976
9355    977
9356    989
9357    991
Name: numero_processo, Length: 9358, dtype: int64>

In [ ]:
proc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9358 entries, 0 to 9357
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   incidente              9358 non-null   int64 
 1   classe                 9358 non-null   object
 2   nome_processo          9358 non-null   object
 3   classe_extenso         9358 non-null   object
 4   tipo_processo          9358 non-null   object
 5   liminar                8168 non-null   object
 6   origem                 9335 non-null   object
 7   relator                9287 non-null   object
 8   autor1                 9351 non-null   object
 9   len(partes_total)      9358 non-null   int64 
 10  partes_total           9351 non-null   object
 11  data_protocolo         9358 non-null   object
 12  origem_orgao           9178 non-null   object
 13  lista_assuntos         8945 non-null   object
 14  resumo                 9358 non-null   object
 15  len(andamentos_lista)

#### Transformação da coluna liminar

In [ ]:
# Para obter valores únicos de uma coluna com listas, convertemos para tupla (que é hashable)
print(proc['liminar'].apply(lambda x: tuple(x) if isinstance(x, list) else x).unique())

[None ('MEDIDA LIMINAR',)
 ('MEDIDA LIMINAR', 'CONVERTIDO EM PROCESSO ELETRÔNICO')
 ('ELEITORAL', 'MEDIDA LIMINAR') ('MEDIDA LIMINAR', 'COVID-19')
 ('CONVERTIDO EM PROCESSO ELETRÔNICO',)
 ('MEDIDA LIMINAR', 'CONVERTIDO EM PROCESSO ELETRÔNICO', 'DOCUMENTO(S) ACAUTELADO(S)')
 ('CRIMINAL', 'MEDIDA LIMINAR') ('MEDIDA LIMINAR', 'TUTELA ANTECIPADA')
 ('ELEITORAL', 'MEDIDA LIMINAR', 'CONVERTIDO EM PROCESSO ELETRÔNICO')
 ('MAIOR DE 60 ANOS OU PORTADOR DE DOENÇA GRAVE', 'MEDIDA LIMINAR')
 ('TUTELA ANTECIPADA',) ('ELEITORAL',)
 ('MEDIDA LIMINAR', 'TUTELA PROVISÓRIA') ('TUTELA PROVISÓRIA',)
 ('ELEITORAL', 'MEDIDA LIMINAR', 'COVID-19')
 ('MEDIDA LIMINAR', 'COVID-19', 'TUTELA PROVISÓRIA') ('COVID-19',)
 ('ELEITORAL', 'MEDIDA LIMINAR', 'TUTELA PROVISÓRIA')
 ('MEDIDA LIMINAR', 'LEI MARIA DA PENHA') ('CRIANÇA E ADOLESCENTE (ECA)',)
 ('CRIMINAL', 'MEDIDA LIMINAR', 'TUTELA PROVISÓRIA')
 ('MEDIDA LIMINAR', 'PESSOA COM DEFICIÊNCIA')
 ('MEDIDA LIMINAR', 'DOCUMENTO(S) ACAUTELADO(S)')
 ('MEDIDA LIMINAR', 'CR

#### Transformação da coluna orgão de origem

Para viabilizar a análise descritiva, o campo original orgao_origem passou por uma etapa de engenharia de dados. Os dados brutos apresentavam alta fragmentação, misturando termos genéricos, abreviações e varas específicas (ex: TRIBUNAL REGIONAL FEDERAL coexistindo com 3ª VARA FEDERAL DO DF). Sem um tratamento, essa falta de padronização pulverizaria os resultados, gerando gráficos poluídos e ilegíveis.

A solução adotada foi a criação de uma nova coluna, denominada esfera_origem, estruturada por meio de um algoritmo de varredura textual que agrupa os órgãos em macrocategorias institucionais homogêneas (como Justiça Federal, Justiça Estadual, Justiça Eleitoral), além de unificar os valores nulos sob a etiqueta Não Identificado.

> A decisão de manter a coluna original intacta e criar uma coluna inédita garante a integridade da base de dados. Dessa forma, preserva-se o nível de detalhe original para futuras investigações específicas, enquanto disponibiliza-se uma variável normalizada, limpa e pronta para a extração imediata de gráficos e estatísticas de frequência.

#### Transformação da coluna `lista_assuntos`

#### Conversão de tipos

- incidente → inteiro.
- data_protocolo → tipo datetime (dia/mês/ano). Erros de conversão viram NaT.

Converte colunas com poucos valores únicos para o tipo category. Isso reduz o uso de memória e acelera agrupamentos.

In [ ]:
proc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9358 entries, 0 to 9357
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   incidente              9358 non-null   int64         
 1   classe                 9358 non-null   category      
 2   nome_processo          9358 non-null   object        
 3   classe_extenso         9358 non-null   category      
 4   tipo_processo          9358 non-null   category      
 5   liminar                8168 non-null   object        
 6   origem                 9335 non-null   category      
 7   relator                9287 non-null   category      
 8   autor1                 9351 non-null   object        
 9   len(partes_total)      9358 non-null   int64         
 10  partes_total           9351 non-null   object        
 11  data_protocolo         9358 non-null   datetime64[ns]
 12  origem_orgao           9178 non-null   object        
 13  lis

### 2.2 Verificação de nulos e duplicatas pos processamento

#### Nulos

In [ ]:
# Mostra a quantidade absoluta de nulos em cada coluna
print("Valores Nulos Absolutos:")
#print(proc.isnull().sum())

# BÔNUS: Mostra o percentual (%) de nulos em cada coluna (ótimo para o relatório)
print("\nPercentual de Nulos (%):")
#print((proc.isnull().sum() / len(proc)) * 100)

Valores Nulos Absolutos:

Percentual de Nulos (%):


#### Duplicatas

In [ ]:
# Para verificar duplicatas em um DataFrame que contém listas,
# convertemos para string para que os valores tornem-se 'hasháveis'
# total_linhas_duplicadas = proc.astype(str).duplicated().sum()
# print(f"Total de linhas completamente duplicadas: {total_linhas_duplicadas}")

Ao observar os mesmos lista_assunto no dataset original, observamos que os mesmos são null. Portanto, os 369 valores nulos são devido ao dataset.

### 3. Explosão dos JSONs aninhados

#### Algoritmo de explosão dos jsons
A coluna original do dataset apresentava dados estruturados em formato JSON/Lista aninhada, agregando em um único registro múltiplos subcampos cruciais, tais como históricos de andamentos processuais, decisões proferidas, trâmites de deslocamento e o rol completo de partes envolvidas na lide. Sob a ótica da ciência de dados, a manutenção dessas informações em seu estado bruto (strings complexas comprimidas em uma única célula) inviabilizaria a execução de análises descritivas e cruzamentos estatísticos fundamentais.  

O processo de "explosão" e normalização desses JSONs foi imperativo pelos seguintes motivos:
- Correção da Relação Um-para-Muitos ($1:N$ : Um único processo judicial frequentemente possui dezenas de decisões e milhares de andamentos. A expansão estruturada permitiu separar cada subevento em uma linha autônoma, viabilizando a contagem exata e o cálculo de frequências reais sem mascarar os dados.  
- Acesso às Variáveis Internas para Correlação: Dados analíticos vitais — como o nome do julgador de uma decisão específica , a data de um andamento ou o papel jurídico de uma parte  — estavam inacessíveis para algoritmos de agregação padrão (como agrupamentos e tabulações cruzadas). A extração converteu chaves de dicionários em colunas explícitas e tipadas.  
- Sanitização de Ruídos e Nulos Ocultos: A varredura textual durante a explosão permitiu identificar e tratar strings vazias ou marcadores genéricos de omissão (como os termos 'NA' e 'nan'), convertendo-os no padrão nulo interpretável pelo sistema (None/NaN).

In [ ]:
from src.cleaning import explode_partes, explode_andamentos, explode_decisoes, explode_deslocamentos

Essa transformação foi o passo fundamental para converter um amontoado de textos semiestruturados em variáveis quantificáveis, permitindo que o relatório final apresente métricas precisas sobre o comportamento interno, fluxos e decisões dos processos analisados.

#### Execução do algoritmo e criação dos 4 novos datasets

In [ ]:
df_partes = explode_partes(proc)

Este conjunto de dados armazena o rol de atores, entidades e defensores vinculados a cada ação judicial.

| Variável | Tipo de Dado | Descrição |
| --- | --- | --- |
| `incidente` | Texto (Chave) | Identificador único do processo no tribunal (usado para realizar o cruzamento com a Tabela Fato). |
| `classe` | Texto | Classe jurídica de controle concentrado (ADI, ADC, ADO, ADPF). |
| `tipo_processo` | Texto | Informa o suporte do processo (Físico ou Eletrônico). |
| `par__index` | Inteiro | Identificador sequencial interno da posição da parte na lista do processo.|
| `par_tipo` | Texto | Polo ou papel jurídico ocupado pela parte (ex: REQTE. para Requerente, ADV. para Advogado).|
| `par_nome` | Texto | Nome completo da pessoa, órgão, agremiação ou conselho profissional, seguido pelas inscrições da OAB no caso de advogados.|

In [ ]:
df_decisoes = explode_decisoes(proc)


Este conjunto de dados isola todos os pronunciamentos judiciais com carga decisória proferidos ao longo da tramitação.

| Variável | Tipo de Dado | Descrição |
| --- | --- | --- |
| `incidente` | Texto (Chave) | Identificador único do processo no tribunal (usado para realizar o cruzamento com a Tabela Fato). |
| `classe` | Texto | Classe jurídica de controle concentrado (ADI, ADC, ADO, ADPF). |
| `tipo_processo` | Texto | Informa o suporte do processo (Físico ou Eletrônico). |
| `dec_index` | Inteiro | Código indexador que correlaciona a decisão ao respectivo evento na tabela de andamentos.|
| `dec_data` | Datetime / Texto | Data em que a decisão foi assinada, publicada ou homologada.|
| `dec_nome` | Texto | Tipificação jurídica do provimento judicial (ex: Convertido em diligência, Despacho, Decisão Referendada, Julgamento Virtual).|
| `dec_complemento` | Texto | Resumo explicativo ou descrição curta inserida pelo gabinete judicial sobre o comando exarado.|
| `dec_julgador` | Texto | Autoridade prolatora do ato, identificando se foi uma decisão monocrática de um Ministro Relator ou um acórdão de órgão colegiado.|
| `dec_validade` | Texto | Validação sistêmica da publicação da peça decisória.|
| `dec_link` | Texto | Endereço eletrônico direto para recuperação do inteiro teor em formato PDF ou RTF no repositório do tribunal.|
| `dec_link_tipo` | Texto | Classificação de formato técnico do link gerado.|
| `dec_link_conteúdo` | Texto | Transcrição textual completa dos fundamentos e do dispositivo da decisão (utilizado para processamento de linguagem natural).|

In [ ]:
df_deslocamentos = explode_deslocamentos(proc)

Este conjunto rastreia o trâmite logístico e físico/eletrônico dos autos entre os diferentes setores, gerências e secretarias do tribunal.

| Variável | Tipo de Dado | Descrição |
| --- | --- | --- |
| `incidente` | Texto (Chave) | Identificador único do processo no tribunal (usado para realizar o cruzamento com a Tabela Fato). |
| `classe` | Texto | Classe jurídica de controle concentrado (ADI, ADC, ADO, ADPF). |
| `tipo_processo` | Texto | Informa o suporte do processo (Físico ou Eletrônico). |
| `des_index` | Inteiro | Número sequencial que ordena as movimentações de carga do processo.|
| `des_data_recebido` | Texto | Registro da data em que a unidade de destino oficializou o recebimento do processo em seu setor.|
| `des_enviado por` | Texto | Nome da secretaria, subsecretaria ou gerência de origem que despachou o processo.|
| `des_recebido por` | Texto | Nome do departamento administrativo de destino encarregado de absorver o processo, constando frequentemente detalhes adicionais da remessa.|
| `des_guia` | Texto | Código numérico oficial da Guia de Remessa gerada para controle de auditoria e protocolo logístico do tribunal.|

In [ ]:
df_andamentos = explode_andamentos(proc)

Este conjunto mapeia a linha do tempo completa e o histórico de movimentações processuais do início até a última atualização.

| Variável | Tipo de Dado | Descrição |
| --- | --- | --- |
| `incidente` | Texto (Chave) | Identificador único do processo no tribunal (usado para realizar o cruzamento com a Tabela Fato). |
| `classe` | Texto | Classe jurídica de controle concentrado (ADI, ADC, ADO, ADPF). |
| `tipo_processo` | Texto | Informa o suporte do processo (Físico ou Eletrônico). |
| `and_index` | Inteiro | Número indexador do andamento na ordem cronológica (geralmente inversa).|
| `and_data` | Datetime / Texto | Data exata em que o evento ou certidão foi lançado no sistema do tribunal.|
| `and_nome` | Texto | Título descritivo do ato processual praticado (ex: Certidão, Expedido, Conclusos).|
| `and_complemento` | Texto | Detalhamento em texto corrido sobre o andamento (ex: conteúdo de certidões, números de ofícios e destinatários de remessas).|
| `and_julgador` | Texto | Órgão ou autoridade associada ao ato. Recebe o valor 'NA' quando se trata de um ato puramente administrativo de secretaria.|
| `and_validade` | Texto | Indicador interno de integridade ou validação do registro no banco de dados (ex: 'valid').|
| `and_link` | Texto | URL para o download do documento ou peça jurídica correspondente no portal oficial do STF.|
| `and_link_tipo` | Texto | Metadado que categoriza a extensão ou o formato do arquivo indexado.|
| `and_link_conteúdo` | Texto | Texto bruto extraído digitalmente ou via OCR das páginas do documento atrelado ao andamento.|


### 4. Datasets finais

#### Dataset antigo (dadosConcatenados)
Com a conclusão da extração e normalização dos dados aninhados em quatro datasets satélites independentes, aplicou-se o princípio de segregação de responsabilidades na base de dados. As colunas originais de JSON bruto (partes_total, andamentos, decisões e deslocamento) foram permanentemente removidas do dataset principal.

In [ ]:
from src.cleaning import build_fact_table

proc = build_fact_table(proc)

Remove colunas JSON brutas (já exportadas separadamente).

Essa abordagem converteu a arquitetura do projeto em um modelo analítico estruturado (frequentemente denominado Star Schema ou Esquema Estrela). O dataset original passou a atuar estritamente como uma Tabela Fato, centralizando os metadados gerais do processo (como classe, relator e esferas de origem), enquanto as novas tabelas funcionam como Dimensões detalhadas conectadas de forma relacional pela chave única do incidente. Esta decisão metodológica eliminou a redundância severa de armazenamento, reduziu drasticamente o consumo de memória RAM no ambiente de execução e otimizou a performance computacional para a geração dos gráficos e cruzamentos estatísticos subsequentes.

#### Validação

In [ ]:
print("=== VALIDACAO DE QUALIDADE DOS DADOS ===")

# 1. VALIDACAO DA TABELA FATO (processos_final)
print("\n[TABELA FATO: PROCESSOS]")

# Sem duplicatas de incidente
dupl = proc["incidente"].duplicated().sum()
if dupl == 0:
    print(f"  Incidentes duplicados: {dupl} - OK")
else:
    print(f"  CRITICO: {dupl} incidentes duplicados detectados!")

# Datas dentro do esperado (1988 – hoje)
fora_range = proc[
    (proc["data_protocolo"].dt.year < 1988)
    | (proc["data_protocolo"].dt.year > 2026)
]
if len(fora_range) == 0:
    print(f"  Datas fora do range (1988-2026): {len(fora_range)} - OK")
else:
    print(
        f"  AVISO: {len(fora_range)} processos com datas fora do range esperado!"
    )

# Classes esperadas do Controle Concentrado
classes_permitidas = {"ADI", "ADC", "ADO", "ADPF"}
classes_encontradas = set(proc["classe"].unique())
classes_ok = classes_encontradas <= classes_permitidas
if classes_ok:
    print("  Classes restritas ao escopo (ADI/ADC/ADO/ADPF): OK")
else:
    print("  AVISO: Classes extras detectadas no dataset!")
    print(f"    Classes inesperadas: {classes_encontradas - classes_permitidas}")

# tipo_processo so Fisico/Eletronico
tipos_ok = set(proc["tipo_processo"].unique()) <= {
    "Físico",
    "Eletrônico",
}
if tipos_ok:
    print("  Meio processual (Fisico/Eletronico apenas): OK")
else:
    print("  AVISO: Valores invalidos encontrados em tipo_processo!")


# 2. VALIDACAO DE INTEGRIDADE REFERENCIAL (DATASETS FILHOS)
print("\n[TABELAS FILHAS: INTEGRIDADE REFERENCIAL]")

dimensoes = {
    "Partes": df_partes,
    "Andamentos": df_andamentos,
    "Decisoes": df_decisoes,
    "Deslocamentos": df_deslocamentos,
}

incidentes_fato = set(proc["incidente"])

for nome, df_dim in dimensoes.items():
    total_linhas = len(df_dim)

    if total_linhas == 0:
        print(f"  {nome}: Alerta: Tabela vazia!")
        continue

    # Encontra incidentes na dimensao que NAO existem na fato
    incidentes_dim = set(df_dim["incidente"].dropna())
    orfaos = incidentes_dim - incidentes_fato

    if len(orfaos) == 0:
        print(
            f"  {nome} -> Total Linhas: {total_linhas} | Chaves Estrangeiras: OK"
        )
    else:
        print(
            f"  {nome} -> Total Linhas: {total_linhas} | CRITICO: {len(orfaos)} incidentes orfaos!"
        )
        print(f"    Primeiros IDs orfaos: {list(orfaos)[:5]}")

print("\n=== FIM DA VALIDACAO ===")

=== VALIDACAO DE QUALIDADE DOS DADOS ===

[TABELA FATO: PROCESSOS]
  Incidentes duplicados: 0 - OK
  Datas fora do range (1988-2026): 0 - OK
  Classes restritas ao escopo (ADI/ADC/ADO/ADPF): OK
  Meio processual (Fisico/Eletronico apenas): OK

[TABELAS FILHAS: INTEGRIDADE REFERENCIAL]


NameError: name 'df_andamentos' is not defined

### 5. Exportação para `parquet`

Na etapa final do pipeline de tratamento, os cinco datasets resultantes (a Tabela Fato de metadados gerais e as quatro Tabelas de Dimensão oriundas da expansão dos JSONs) foram exportados utilizando o formato Apache Parquet (.parquet).

A escolha do Parquet em detrimento de formatos textuais convencionais (como o .csv) fundamenta-se em eficiência arquitetural. Por ser um formato de armazenamento estritamente colunar e binário, ele aplica algoritmos de compressão altamente eficientes, reduzindo drasticamente o espaço físico ocupado em disco. Adicionalmente, o Parquet preserva integralmente os tipos de dados originais das colunas definidos em memória (metadados), eliminando a necessidade de reprocessamento ou parsing de tipos (como datas e identificadores) em carregamentos futuros, otimizando o tempo de leitura para as fases subsequentes de modelagem analítica e visualização de dados.

In [ ]:
datasets = {
    #'arquivosConcatenados': proc,
    #'dim_partes': df_partes,
    'dim_andamentos': df_andamentos,
    #'dim_decisoes': df_decisoes,
    #'dim_deslocamentos': df_deslocamentos,
}

# Garante que o diretório de destino exista antes de exportar
os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Iniciando a exportação para Parquet...")
for nome_arquivo, data_frame in datasets.items():
    caminho_completo = os.path.join(PROCESSED_PATH, f'{nome_arquivo}.parquet')

    # Exporta usando a engine padrão (pyarrow ou fastparquet)
    data_frame.to_parquet(caminho_completo, index=False)
    print(f' -> Salvo com sucesso: {caminho_completo}')

print("\nTodos os arquivos foram exportados com sucesso!")

Iniciando a exportação para Parquet...
 -> Salvo com sucesso: data/processed/dim_andamentos.parquet

Todos os arquivos foram exportados com sucesso!


### Visualizações dos novos datasets

In [ ]:
proc

,incidente,classe,nome_processo,classe_extenso,tipo_processo,liminar,origem,relator,autor1,len(partes_total),data_protocolo,origem_orgao,lista_assuntos,resumo,len(andamentos_lista),len(decisões),len(deslocamentos),status_processo,numero_processo,esfera_origem
0,2221383,ADC,ADC 10,AÇÃO DECLARATÓRIA DE CONSTITUCIONALIDADE,Físico,None,RJ,AYRES BRITTO,DINETE LESSA,2,2004-05-18,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO CIVIL | Coisas | Enfiteuse""]","<div class=""col-x...",7,0,14,Finalizado,10,Justiça Federal
1,2338671,ADC,ADC 11,AÇÃO DECLARATÓRIA DE CONSTITUCIONALIDADE,Eletrônico,[MEDIDA LIMINAR],DF,GILMAR MENDES,GOVERNADOR DO DISTRITO FEDERAL,9,2005-11-28,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO PROCESSUAL CIVIL E DO TRABALHO | Liq...","<div class=""col-x...",128,6,65,Finalizado,11,Justiça Federal
2,2358461,ADC,ADC 12,AÇÃO DECLARATÓRIA DE CONSTITUCIONALIDADE,Físico,[MEDIDA LIMINAR],DF,AYRES BRITTO,ASSOCIAÇÃO DOS MAGISTRADOS BRASILEIROS - AMB,16,2006-02-02,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO PROCESSUAL CIVIL E DO TRABALHO | Órg...","<div class=""col-x...",115,2,49,Finalizado,12,Justiça Federal
3,2383568,ADC,ADC 13,AÇÃO DECLARATÓRIA DE CONSTITUCIONALIDADE,Físico,None,DF,JOAQUIM BARBOSA,ASSOCIAÇÃO BRASILEIRA DAS EMPRESAS DE TRADING ...,3,2006-05-24,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO TRIBUTÁRIO | Impostos | IPI/ Imposto...","<div class=""col-x...",11,0,13,Finalizado,13,Justiça Federal
4,2415705,ADC,ADC 14,AÇÃO DECLARATÓRIA DE CONSTITUCIONALIDADE,Eletrônico,[MEDIDA LIMINAR],DF,ROSA WEBER,ASSOCIAÇÃO DOS NOTÁRIOS E REGISTRADORES DO BRA...,12,2006-09-20,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...","<div class=""col-x...",204,23,75,Finalizado,14,Justiça Federal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9353,6404537,ADPF,ADPF 973,ARGUIÇÃO DE DESCUMPRIMENTO DE PRECEITO FUNDAME...,Eletrônico,"[MEDIDA LIMINAR, PROCESSO ESTRUTURAL]",DF,LUIZ FUX,PARTIDO DOS TRABALHADORES,90,2022-05-13,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...","<div class=""col-x...",193,11,24,Em andamento,973,Justiça Federal
9354,6410647,ADPF,ADPF 976,ARGUIÇÃO DE DESCUMPRIMENTO DE PRECEITO FUNDAME...,Eletrônico,"[MEDIDA LIMINAR, PROCESSO ESTRUTURAL]",DF,ALEXANDRE DE MORAES,REDE SUSTENTABILIDADE,106,2022-05-23,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...","<div class=""col-x...",761,8,79,Em andamento,976,Justiça Federal
9355,6414142,ADPF,ADPF 977,ARGUIÇÃO DE DESCUMPRIMENTO DE PRECEITO FUNDAME...,Eletrônico,[MEDIDA LIMINAR],DF,ANDRÉ MENDONÇA,PARTIDO SOCIALISTA BRASILEIRO,6,2022-05-26,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...","<div class=""col-x...",23,1,10,Em andamento,977,Justiça Federal
9356,6437138,ADPF,ADPF 989,ARGUIÇÃO DE DESCUMPRIMENTO DE PRECEITO FUNDAME...,Eletrônico,[MEDIDA LIMINAR],DF,LUÍS ROBERTO BARROSO,SOCIEDADE BRASILEIRA DE BIOETICA - SBB,76,2022-06-30,SUPREMO TRIBUNAL FEDERAL,"[""DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE ...","<div class=""col-x...",182,8,29,Em andamento,989,Justiça Federal


In [ ]:
df_partes

,incidente,classe,tipo_processo,par__index,par_tipo,par_nome
0,2221383,ADC,Físico,1,REQTE.(S),DINETE LESSA
1,2221383,ADC,Físico,2,ADV.(A/S),ADRIANO FERNANDES
2,2338671,ADC,Eletrônico,1,REQTE.(S),GOVERNADOR DO DISTRITO FEDERAL
3,2338671,ADC,Eletrônico,2,ADV.(A/S),PROCURADOR-GERAL DO DISTRITO FEDERAL
4,2338671,ADC,Eletrônico,3,AM. CURIAE.,UNIÃO
...,...,...,...,...,...,...
70979,6437270,ADPF,Eletrônico,36,ADV.(A/S),"BEATRIZ MENDONCA DA COSTA (75474/DF, 229218/RJ)"
70980,6437270,ADPF,Eletrônico,37,AM. CURIAE.,OBSERVATÓRIO DOS DIREITOS HUMANOS DOS POVOS IN...
70981,6437270,ADPF,Eletrônico,38,ADV.(A/S),LUCAS CRAVO DE OLIVEIRA (65829/DF)
70982,6437270,ADPF,Eletrônico,39,ADV.(A/S),CAROLINA RIBEIRO SANTANA (66511/DF)


In [ ]:
df_andamentos

,incidente,classe,tipo_processo,and_index,and_data,and_nome,and_complemento,and_julgador,and_validade,and_link,and_link_tipo,and_link_conteúdo
0,2221383,ADC,Físico,7,16/06/2004,BAIXA AO ARQUIVO DO STF,GUIA 6361,None,valid,None,None,None
1,2221383,ADC,Físico,6,11/06/2004,REMESSA DOS AUTOS,À SEÇÃO DE BAIXA DE PROCESSOS,None,valid,None,None,None
2,2221383,ADC,Físico,5,11/06/2004,DECORRIDO O PRAZO,EM 07/06/04 SEM QUE FOSSE INTERPOSTO RECURSO D...,None,valid,None,None,None
3,2221383,ADC,Físico,4,31/05/2004,"PUBLICACAO, DJ:",DA DECISÃO DE 21.05.04,None,valid,None,None,None
4,2221383,ADC,Físico,3,25/05/2004,DECISÃO DO(A) RELATOR(A) - NEGADO SEGUIMENTO,"EM 21.05.04 ""À LUZ DO ART. 103, § 4º, DA LEI D...",None,valid,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
505144,6437270,ADPF,Eletrônico,5,30/06/2022,Despacho,Tendo em vista o requerimento de distribuição ...,None,valid,None,None,None
505145,6437270,ADPF,Eletrônico,4,30/06/2022,Conclusos ao(à) Relator(a),None,None,valid,None,None,None
505146,6437270,ADPF,Eletrônico,3,30/06/2022,Distribuído,MIN. EDSON FACHIN,None,valid,https://portal.stf.jus.br/processos/downloadPe...,None,Supremo Tribunal Federal\nTERMO DE RECEBIMENTO...
505147,6437270,ADPF,Eletrônico,2,30/06/2022,Autuado,None,None,valid,None,None,None


In [ ]:
df_decisoes

,incidente,classe,tipo_processo,dec_index,dec_data,dec_nome,dec_complemento,dec_julgador,dec_validade,dec_link,dec_link_tipo,dec_link_conteúdo
0,2338671,ADC,Eletrônico,117,23/08/2019,Procedente,"Decisão: O Tribunal, por maioria, conheceu da ...",TRIBUNAL PLENO - SESSÃO VIRTUAL,valid,https://portal.stf.jus.br/processos/downloadTe...,None,"Decisão: O Tribunal, por maioria, conheceu da ..."
1,2338671,ADC,Eletrônico,113,31/07/2019,Inclua-se em pauta - minuta extraída,Julgamento Virtual: . Incluído na Lista 24-201...,TRIBUNAL PLENO - SESSÃO VIRTUAL,valid,None,None,None
2,2338671,ADC,Eletrônico,59,26/01/2011,Indeferido,o pedido de Anildo Fabio de Araujo e DEFERIDO ...,MIN. GILMAR MENDES,valid,None,None,None
3,2338671,ADC,Eletrônico,56,14/01/2011,Deferido,None,MIN. GILMAR MENDES,invalid,None,None,None
4,2338671,ADC,Eletrônico,29,26/08/2009,Questão de ordem,"Decisão: Decisão: O Tribunal, por maioria e no...",TRIBUNAL PLENO,valid,https://portal.stf.jus.br/processos/downloadTe...,None,"Decisão: O Tribunal, por maioria e nos termos ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
30137,6437270,ADPF,Eletrônico,70,13/12/2022,Destaque do(a) Ministro(a),Decisão: Após os votos dos Ministros Edson Fac...,MIN. NUNES MARQUES,valid,https://portal.stf.jus.br/processos/downloadTe...,None,Decisão: Após os votos dos Ministros Edson Fac...
30138,6437270,ADPF,Eletrônico,69,12/12/2022,Processo destacado no Julgamento Virtual,Pedido de Destaque. Sessão de 02/12/2022 a 12/...,MIN. NUNES MARQUES,valid,None,None,None
30139,6437270,ADPF,Eletrônico,47,22/11/2022,Inclua-se em pauta - minuta extraída,Julgamento Virtual: ADPF-MC-Ref. Incluído na L...,TRIBUNAL PLENO - SESSÃO VIRTUAL,valid,None,None,None
30140,6437270,ADPF,Eletrônico,40,21/11/2022,Liminar deferida ad referendum,Articulação dos Povos Indígenas do Brasil – AP...,MIN. EDSON FACHIN,valid,None,None,None


In [ ]:
df_deslocamentos

,incidente,classe,tipo_processo,des_index,des_data_recebido,des_enviado por,des_recebido por,des_guia
0,2221383,ADC,Físico,14,Recebido em 30/01/2021,"COORDENADORIA DE GESTÃO DA INFORMAÇÃO, MEMÓRIA...",Enviado por COORDENADORIA DE MEMÓRIA E GESTÃO ...,Guia 6/2021
1,2221383,ADC,Físico,13,Recebido em 14/06/2019,COORDENADORIA DE MEMÓRIA E GESTÃO DOCUMENTAL,Enviado por SEÇÃO DE ARQUIVO em 14/06/2019,Guia 399/2019
2,2221383,ADC,Físico,12,Recebido em 11/11/2014,SEÇÃO DE ARQUIVO,Enviado por SEÇÃO DE ARQUIVO em 11/11/2014,Guia 312/2014
3,2221383,ADC,Físico,11,Recebido em 06/06/2012,SEÇÃO DE ARQUIVO,Enviado por SEÇÃO DE ARQUIVO em 06/06/2012,Guia 35/2012
4,2221383,ADC,Físico,10,Recebido em 02/05/2012,SEÇÃO DE ARQUIVO,Enviado por SEÇÃO DE ARQUIVO em 02/05/2012,Guia 158/2012
...,...,...,...,...,...,...,...,...
251950,6437270,ADPF,Eletrônico,5,Recebido em 01/07/2022,GERÊNCIA DE CONTROLE CONCENTRADO E RECLAMAÇÕES,Enviado por PRESIDÊNCIA em 01/07/2022,Guia 22369/2022
251951,6437270,ADPF,Eletrônico,4,Recebido em 30/06/2022,PRESIDÊNCIA,Enviado por GERÊNCIA DE CONTROLE CONCENTRADO E...,Guia 11235/2022
251952,6437270,ADPF,Eletrônico,3,Recebido em 30/06/2022,GERÊNCIA DE CONTROLE CONCENTRADO E RECLAMAÇÕES,Enviado por GABINETE MINISTRO EDSON FACHIN em ...,Guia 4010/2022
251953,6437270,ADPF,Eletrônico,2,Recebido em 30/06/2022,GABINETE MINISTRO EDSON FACHIN,"Enviado por GERÊNCIA DE AUTUAÇÃO, ANÁLISE DE P...",Guia 4005/2022
